In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

In [9]:
import importlib
import config

importlib.reload(config)

<module 'config' from 'd:\\proposal\\thiese\\code\\Persian-Topic-Modeling\\src\\config.py'>

In [11]:
import pandas as pd
import json
from sklearn.model_selection import train_test_split
from config import(
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    TEST_SIZE,
    RANDOM_SEED_SPLIT,
    TEST_DATA_PATH,
    TRAIN_DATA_PATH,
    SPLIT_INFO
)
from preprocessing import clean_text


In [ ]:
df = pd.read_csv(RAW_DATA_DIR/ "isna_complete.csv")

# Remove missing documents
df = (
    df.dropna(subset=["content"])
      .reset_index(drop=True)
)

# Clean text
df["content"] = (
    df["content"]
      .astype(str)
      .apply(clean_text)
)

# Remove empty documents after preprocessing
df = (
    df[df["content"].str.strip() != ""]
      .reset_index(drop=True)
)

df.to_csv(
    PROCESSED_DATA_DIR / "news_clean.csv",
    index=False,
    encoding="utf-8-sig"
)



In [8]:
clean_data = pd.read_csv(PROCESSED_DATA_DIR / "news_clean.csv")

train_df, test_df = train_test_split(
    clean_data,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED_SPLIT,
    stratify=clean_data['category']
)

# ذخیره
train_df.to_csv(TRAIN_DATA_PATH, index=False)
test_df.to_csv(TEST_DATA_PATH, index=False)

print(f"Train size: {len(train_df)}")
print(f"Test size: {len(test_df)}")

Train size: 4775
Test size: 1194


In [16]:
split_info = {
    'total_documents': len(clean_data),
    'train_size': len(train_df),
    'test_size': len(test_df),
    'test_ratio': TEST_SIZE,
    'random_state': RANDOM_SEED_SPLIT,
    'stratify_column': 'category',  # ✅ نام ستون به صورت رشته
    'train_distribution': train_df['category'].value_counts().to_dict(),  # ✅ استفاده از train_df
    'test_distribution': test_df['category'].value_counts().to_dict()  # ✅ استفاده از test_df
}

with open(SPLIT_INFO, 'w', encoding='utf-8') as f:
    json.dump(split_info, f, ensure_ascii=False, indent=2)

print(f"\n✅ داده‌ها با موفقیت تقسیم شدند:")
print(f"   - Train: {len(train_df)} سند ({len(train_df)/len(clean_data)*100:.1f}%)")
print(f"   - Test: {len(test_df)} سند ({len(test_df)/len(clean_data)*100:.1f}%)")
print(f"\n📁 فایل‌ها در {SPLIT_INFO} ذخیره شدند.")


✅ داده‌ها با موفقیت تقسیم شدند:
   - Train: 4775 سند (80.0%)
   - Test: 1194 سند (20.0%)

📁 فایل‌ها در D:\proposal\thiese\code\Persian-Topic-Modeling\data\splits\split_info.json ذخیره شدند.
